In [0]:
df_customers=spark.read.format("csv")\
    .option("header",True)\
    .option("inferschema",True)\
    .load("/Volumes/pysparkdbt/source_data/source/customers")

In [0]:
df_customers.Schema

In [0]:
print(df_customers.schema)

In [0]:
schema_customer=df_customers.schema


In [0]:

entities=["customers","drivers","location","trips","vehicles","payments"]

In [0]:

for entity in entities:
    df_batch=spark.read.format("csv")\
    .option("header",True)\
    .option("inferschema",True)\
    .load(f"/Volumes/pysparkdbt/source_data/source/{entity}")

    schema_entity=df_batch.schema


    df_stream=spark.readStream.format("csv")\
        .option("header",True)\
        .schema(schema_entity)\
        .load(f"/Volumes/pysparkdbt/source_data/source/{entity}")

    df_stream.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointlocation",f"/Volumes/pysparkdbt/bronze/bronze_ingestion/{entity}")\
        .trigger(once=True)\
        .toTable(f"pysparkdbt.bronze.{entity}")
